# Joint Communication and Sensing (ISAC)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jman4162/metasurface-py/blob/main/notebooks/04_sensing_isac.ipynb)

This notebook demonstrates:
1. Radar detection SNR computation
2. Localization accuracy (CRLB / Fisher Information)
3. Joint comms-sensing objective
4. How different metasurface configurations trade off communication vs sensing performance

In [ ]:
# !pip install metasurface-py

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from metasurface_py.geometry import RectangularLattice
from metasurface_py.elements import PhaseOnlyCell
from metasurface_py.elements.states import ContinuousPhaseSpace
from metasurface_py.surfaces import Metasurface
from metasurface_py.core.types import Position3D, AngleGrid
from metasurface_py.em import steering_phase, far_field_pattern
from metasurface_py.sensing import (
    detection_snr, fisher_information_matrix, crlb_position,
    monostatic_rcs,
)
from metasurface_py.plotting import set_publication_style, plot_pattern_2d

set_publication_style(target="ieee_double")

## Setup: Dual-Function Metasurface

A 16×16 metasurface at 10 GHz serves both communication (beam toward user) and sensing (radar toward target).

In [ ]:
freq = 10e9
lam = 3e8 / freq
lattice = RectangularLattice(nx=16, ny=16, dx=lam/2, dy=lam/2)
cell = PhaseOnlyCell(state_space=ContinuousPhaseSpace())
surface = Metasurface(lattice=lattice, cell=cell)

# Comm user at 20° and radar target at 50°
comm_theta = np.radians(20)
radar_theta = np.radians(50)
target_pos = Position3D(x=50*np.sin(radar_theta), y=0.0, z=50*np.cos(radar_theta))

print(f"Comm direction: θ = 20°")
print(f"Radar target: θ = 50°, range = 50 m")

## Radar Detection SNR

We compute the radar detection SNR when the beam is steered toward the target.

In [ ]:
phase_radar = steering_phase(lattice, radar_theta, 0.0, freq)
state_radar = surface.set_state(phase_radar)

snr_radar = detection_snr(
    surface, state_radar, target_pos, freq,
    tx_power=1.0, noise_power=1e-12, target_rcs=1.0
)
print(f"Detection SNR (beam at target): {10*np.log10(snr_radar):.1f} dB")

## Localization Accuracy (CRLB)

In [ ]:
fim = fisher_information_matrix(surface, state_radar, target_pos, freq, snr=100)
crlb = crlb_position(surface, state_radar, target_pos, freq, snr=100)

print(f"CRLB (x, y, z): {np.sqrt(crlb) * 1e3} mm")
print(f"Position accuracy (RSS): {np.sqrt(np.sum(crlb))*1e3:.1f} mm")

## Comms vs Sensing Tradeoff

We sweep the steering angle from the comm direction (20°) to the radar direction (50°) and observe the tradeoff.

In [ ]:
steer_angles = np.linspace(comm_theta, radar_theta, 20)
comm_gains = []
radar_snrs = []

angles_eval = AngleGrid(
    theta=np.linspace(0.01, np.pi - 0.01, 180),
    phi=np.linspace(0, 2*np.pi - 0.01, 36),
)

for theta_s in steer_angles:
    phase = steering_phase(lattice, theta_s, 0.0, freq)
    state = surface.set_state(phase)
    
    # Comm gain in user direction
    from metasurface_py.em.array_factor import far_field_pattern, directivity
    pattern = far_field_pattern(surface, state, freq, angles_eval)
    d = directivity(pattern)
    theta_idx = np.argmin(np.abs(angles_eval.theta - comm_theta))
    comm_gain_dbi = 10 * np.log10(max(float(d.values[theta_idx, 0]), 1e-10))
    comm_gains.append(comm_gain_dbi)
    
    # Radar SNR
    snr = detection_snr(surface, state, target_pos, freq,
                        tx_power=1.0, noise_power=1e-12)
    radar_snrs.append(10 * np.log10(max(snr, 1e-30)))

steer_deg = np.rad2deg(steer_angles)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))

# Tradeoff curves
ax1.plot(steer_deg, comm_gains, 'o-', label='Comm Gain [dBi]')
ax1.plot(steer_deg, radar_snrs, 's--', label='Radar SNR [dB]')
ax1.axvline(x=20, color='blue', alpha=0.3, linestyle=':')
ax1.axvline(x=50, color='orange', alpha=0.3, linestyle=':')
ax1.set_xlabel('Steering Angle [deg]')
ax1.set_ylabel('Performance [dB]')
ax1.set_title('Comms vs Sensing Tradeoff')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Pareto-like scatter
ax2.scatter(comm_gains, radar_snrs, c=steer_deg, cmap='viridis', s=50)
ax2.set_xlabel('Comm Gain [dBi]')
ax2.set_ylabel('Radar Detection SNR [dB]')
ax2.set_title('ISAC Tradeoff Space')
cbar = plt.colorbar(ax2.collections[0], ax=ax2)
cbar.set_label('Steering Angle [deg]')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()